In [1]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

In [2]:
!pip install clean-text

# SEMANTIC SIMILARITY

In [3]:
%%writefile constants.py
EMBDEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"

# https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"

CLEAN_TEXT = True
TOP_K = 500
BATCH_SIZE = 4  # Increased from 1 to 4 for better GPU utilization with 2 GPUs

Writing constants.py


In [4]:
%%writefile utils.py
import pandas as pd
import torch.distributed as dist

from datasets import Dataset
from cleantext import clean
from tqdm.auto import tqdm

from constants import CLEAN_TEXT


def build_prompt(row):
    return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""


def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )



def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)
            
    dataframe = pd.concat(flatten, axis=0)    
    dataframe = dataframe.drop_duplicates(ignore_index=True)
    return dataframe


def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)

    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map(
            {
                1: 1,
                0: -1,
            }
        )

    return dataframe

Writing utils.py


In [5]:
%%writefile sampling.py
import pandas as pd
from constants import DATA_PATH


def get_test_subreddit_proportions():
    """Get subreddit proportions for each rule from test data"""
    test_df = pd.read_csv(f"{DATA_PATH}/test.csv")
    
    proportions = {}
    for rule in test_df["rule"].unique():
        rule_data = test_df[test_df["rule"] == rule]
        subreddit_counts = rule_data["subreddit"].value_counts()
        subreddit_props = subreddit_counts / subreddit_counts.sum()
        proportions[rule] = subreddit_props.to_dict()
    
    return proportions


def sample_unlabelled_data(unlabelled_path, multiplier):
    """Sample unlabelled data maintaining test data proportions"""
    unlabelled_df = pd.read_csv(unlabelled_path)
    proportions = get_test_subreddit_proportions()

    test_df = pd.read_csv(f"{DATA_PATH}/test.csv")
    total_sample_size = len(test_df)* multiplier
    sampled_data = []
    
    # Calculate samples per rule based on test data rule distribution
    rule_counts = test_df["rule"].value_counts()
    rule_proportions = rule_counts / rule_counts.sum()

    unlabelled_df.drop(unlabelled_df[unlabelled_df['body'].str.len() > 2000].index, inplace=True)
    #filter out body present in test,examples
    excluded_values = test_df[['body', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2']].values.ravel()#+ train_df[['body', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2']].values.ravel()
    unlabelled_df = unlabelled_df.query('body not in @excluded_values')
    
    for rule, rule_prop in rule_proportions.items():
        rule_sample_size = int(total_sample_size * rule_prop)
        subreddit_props = proportions[rule]
        
        rule_samples = []
        for subreddit, sub_prop in subreddit_props.items():
            subreddit_sample_size = int(rule_sample_size * sub_prop)
            
            # Sample from unlabelled data for this subreddit
            subreddit_data = unlabelled_df[unlabelled_df["subreddit"].str.lower().str.strip() == subreddit.lower().strip()]
            if len(subreddit_data) ==0 or subreddit_sample_size==0: 
                # Fallback: sample from any data for this rule
                continue
                # subreddit_data = unlabelled_df[unlabelled_df["rule"] == rule]

            if len(subreddit_data) >= subreddit_sample_size:
                sampled = subreddit_data.sample(n=subreddit_sample_size, random_state=42)
            else:
                # If not enough data, sample with replacement
                sampled = subreddit_data.sample(n=subreddit_sample_size, replace=True, random_state=42)
            
            sampled = sampled.copy()
            sampled["rule"] = rule
            rule_samples.append(sampled)
        
        if rule_samples:
            sampled_data.append(pd.concat(rule_samples, axis=0))
    
    # Combine all samples and shuffle
    final_sample = pd.concat(sampled_data, axis=0).reset_index(drop=True)
    final_sample = final_sample.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return final_sample

Writing sampling.py


In [6]:
%%writefile semantic_unlabelled.py
import pandas as pd
import gc
import torch
import torch.multiprocessing as mp
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm
import threading
from concurrent.futures import ThreadPoolExecutor

from utils import get_dataframe_to_train, prepare_dataframe
from sampling import sample_unlabelled_data
from constants import DATA_PATH, EMBDEDDING_MODEL_PATH, EMBEDDING_MODEL_QUERY, TOP_K, BATCH_SIZE


def create_embedding_model(device_id):
    """Create embedding model on specific GPU"""
    return SentenceTransformer(
        model_name_or_path=EMBDEDDING_MODEL_PATH,
        device=f"cuda:{device_id}",
    )


def encode_batch_on_gpu(model, sentences, prompt, batch_size, device_id):
    """Encode sentences on specific GPU"""
    return model.encode(
        sentences=sentences,
        prompt=prompt if prompt else None,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_tensor=True,
        device=f"cuda:{device_id}",
        normalize_embeddings=True,
    )


def get_unlabelled_scores_multi_gpu(unlabelled_dataframe):
    """Generate similarity scores for unlabelled data using training corpus with 2 GPUs"""
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)
    
    # Check available GPUs
    num_gpus = min(2, torch.cuda.device_count())
    print(f"Using {num_gpus} GPUs")
    
    if num_gpus < 2:
        print("Warning: Less than 2 GPUs available, falling back to single GPU")
        return get_unlabelled_scores_single_gpu(unlabelled_dataframe)

    result = []
    for rule in tqdm(unlabelled_dataframe["rule"].unique(), desc="Generate scores for each rule"):
        unlabelled_rule_data = unlabelled_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_rule_data = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_rule_data = corpus_rule_data.reset_index(names="row_id")
        
        # Prepare prompts for unlabelled data
        unlabelled_rule_data = prepare_dataframe(unlabelled_rule_data)
        
        # Create models on both GPUs
        model_gpu0 = create_embedding_model(0)
        model_gpu1 = create_embedding_model(1)
        
        # Split corpus data for document embeddings across both GPUs
        corpus_mid = len(corpus_rule_data) // 2
        corpus_part1 = corpus_rule_data[:corpus_mid]
        corpus_part2 = corpus_rule_data[corpus_mid:]
        
        # Encode document embeddings on both GPUs in parallel
        with ThreadPoolExecutor(max_workers=2) as executor:
            future1 = executor.submit(
                encode_batch_on_gpu, 
                model_gpu0, 
                corpus_part1["prompt"].tolist(), 
                None, 
                BATCH_SIZE, 
                0
            )
            future2 = executor.submit(
                encode_batch_on_gpu, 
                model_gpu1, 
                corpus_part2["prompt"].tolist(), 
                None, 
                BATCH_SIZE, 
                1
            )
            
            doc_embeddings_part1 = future1.result()
            doc_embeddings_part2 = future2.result()
        
        # Combine document embeddings on GPU 0
        document_embeddings = torch.cat([
            doc_embeddings_part1.to("cuda:0"), 
            doc_embeddings_part2.to("cuda:0")
        ], dim=0)
        
        # Process queries in chunks, alternating between GPUs
        chunk_size = 2500  # Smaller chunks for better GPU utilization
        predictions = []
        
        for i in tqdm(range(0, len(unlabelled_rule_data), chunk_size * 2), desc=f"Processing {rule=}"):
            # Get two chunks for parallel processing
            chunk1_end = min(i + chunk_size, len(unlabelled_rule_data))
            chunk2_end = min(i + chunk_size * 2, len(unlabelled_rule_data))
            
            chunk1_queries = unlabelled_rule_data["prompt"][i:chunk1_end].tolist()
            chunk2_queries = unlabelled_rule_data["prompt"][chunk1_end:chunk2_end].tolist() if chunk1_end < chunk2_end else []
            
            # Encode queries on both GPUs in parallel
            with ThreadPoolExecutor(max_workers=2) as executor:
                future1 = executor.submit(
                    encode_batch_on_gpu,
                    model_gpu0,
                    chunk1_queries,
                    EMBEDDING_MODEL_QUERY,
                    BATCH_SIZE,
                    0
                ) if chunk1_queries else None
                
                future2 = executor.submit(
                    encode_batch_on_gpu,
                    model_gpu1,
                    chunk2_queries,
                    EMBEDDING_MODEL_QUERY,
                    BATCH_SIZE,
                    1
                ) if chunk2_queries else None
                
                query_embeddings1 = future1.result().to("cuda:0") if future1 else None
                query_embeddings2 = future2.result().to("cuda:0") if future2 else None
            
            # Process semantic search for both chunks
            for query_embeddings in [query_embeddings1, query_embeddings2]:
                if query_embeddings is None:
                    continue
                    
                semantic_results = semantic_search(
                    query_embeddings,
                    document_embeddings,
                    top_k=TOP_K,
                    score_function=dot_score,
                )
                
                # Process results
                for semantic in semantic_results:
                    semantic_df = pd.DataFrame(semantic)
                    semantic_df = semantic_df.merge(
                        corpus_rule_data[["row_id", "rule_violation"]],
                        how="left", left_on="corpus_id", right_on="row_id",
                    )
                    score = (semantic_df["score"] * semantic_df["rule_violation"]).sum()
                    predictions.append(score)
                    del semantic_df
                
                del semantic_results, query_embeddings
            
            gc.collect()
            torch.cuda.empty_cache()

        unlabelled_rule_data["prediction"] = predictions
        
        # Clean up models
        del model_gpu0, model_gpu1, document_embeddings
        torch.cuda.empty_cache()
        
        # Keep only required columns
        result.append(unlabelled_rule_data[["rule", "body", "prediction"]].copy())
        
    final_results = pd.concat(result, axis=0)
    return final_results


def get_unlabelled_scores_single_gpu(unlabelled_dataframe):
    """Fallback single GPU implementation"""
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)
    
    embedding_model = SentenceTransformer(
        model_name_or_path=EMBDEDDING_MODEL_PATH,
        device="cuda",
    )

    result = []
    for rule in tqdm(unlabelled_dataframe["rule"].unique(), desc="Generate scores for each rule"):
        unlabelled_rule_data = unlabelled_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_rule_data = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_rule_data = corpus_rule_data.reset_index(names="row_id")
        
        # Prepare prompts for unlabelled data
        unlabelled_rule_data = prepare_dataframe(unlabelled_rule_data)
        
        query_embeddings = embedding_model.encode(
            sentences=unlabelled_rule_data["prompt"].tolist(),
            prompt=EMBEDDING_MODEL_QUERY,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        document_embeddings = embedding_model.encode(
            sentences=corpus_rule_data["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        
        chunk_size = 5000  # Process embeddings in chunks
        predictions = []
        for i in tqdm(range(0, len(query_embeddings), chunk_size), desc=f"Processing {rule=}"):
            query_chunk = query_embeddings[i:i+chunk_size]

            semantic_results = semantic_search(
                query_chunk,
                document_embeddings,
                top_k=TOP_K,
                score_function=dot_score,
            )

            for semantic in semantic_results:
                semantic_df = pd.DataFrame(semantic)
                semantic_df = semantic_df.merge(
                    corpus_rule_data[["row_id", "rule_violation"]],
                    how="left", left_on="corpus_id", right_on="row_id",
                )
                score = (semantic_df["score"] * semantic_df["rule_violation"]).sum()
                predictions.append(score)
                del semantic_df

            del semantic_results, query_chunk
            gc.collect()

        unlabelled_rule_data["prediction"] = predictions
        
        # Keep only required columns
        result.append(unlabelled_rule_data[["rule", "body", "prediction"]].copy())
        
    final_results = pd.concat(result, axis=0)
    return final_results


def generate_unlabelled_predictions(unlabelled_csv_path):
    """Main function to generate predictions on unlabelled data"""

    
    # Sample unlabelled data maintaining proportions
    print("Sampling unlabelled data...")
    sampled_unlabelled = sample_unlabelled_data(unlabelled_csv_path, multiplier=5)
    print(f"Sampled {len(sampled_unlabelled)} examples from unlabelled data")
    
    # Generate predictions using multi-GPU
    print("Generating predictions...")
    predictions = get_unlabelled_scores_multi_gpu(sampled_unlabelled)
    
    # Shuffle final results
    predictions = predictions.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Save results
    predictions.to_csv("unlabelled_predictions.csv", index=False)
    print(f"Saved {len(predictions)} predictions to unlabelled_predictions.csv")
    
    return predictions


if __name__ == "__main__":
    unlabelled_csv_path = "/kaggle/input/jigsaw-2m-reddit-unlabelled/reddit-removal-log.csv"  # Update this path as needed
    generate_unlabelled_predictions(unlabelled_csv_path)

Writing semantic_unlabelled.py


In [7]:
%%writefile run_embeddings.py
#!/usr/bin/env python3
import os
import gc
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

from semantic_unlabelled import generate_unlabelled_predictions

if __name__ == "__main__":
  unlabelled_csv_path = "/kaggle/input/jigsaw-2m-reddit-unlabelled/reddit-removal-log.csv"
  predictions = generate_unlabelled_predictions(unlabelled_csv_path)
  print("✅ Embedding processing complete!")
  print(f"Generated predictions for {len(predictions)} samples")
  gc.collect()

Writing run_embeddings.py


In [8]:
import subprocess
import sys

# if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
print("🚀 Running embedding generation in isolated process...")
result = subprocess.run([sys.executable, "run_embeddings.py"],
                    capture_output=True, text=True)

if result.returncode == 0:
  print("✅ Embedding generation completed successfully!")
  print(result.stdout)
else:
  print("❌ Embedding generation failed:")
  print(result.stderr)
  raise RuntimeError("Embedding generation failed")

# Load the results
unlabelled = pd.read_csv('unlabelled_predictions.csv')
print(f"Loaded {len(unlabelled)} predictions")


🚀 Running embedding generation in isolated process...
✅ Embedding generation completed successfully!
Sampling unlabelled data...
Sampled 50 examples from unlabelled data
Generating predictions...
Using 2 GPUs
Saved 50 predictions to unlabelled_predictions.csv
✅ Embedding processing complete!
Generated predictions for 50 samples

Loaded 50 predictions


In [9]:
#unlabelled_predictions.csv ---->rule,body,prediction(not normalizsed)

In [10]:
# CONFIG

SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 10
U_EPOCHS=2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

Using device: cuda


In [11]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [12]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]


# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

Before:(10185, 2)
After: (1875, 2)


,text,label,rule,body,rule_id
0,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ...",0
1,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\n\nplease visit http://www.shifadental.net/te...,0
2,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...,0
3,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...,0
4,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\nFree http://forums.airdroid.com/viewtopic.ph...,0


In [13]:

class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [14]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [15]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    # print(rule_aucs,'Rule_AUC')
    return avg_auc_per_rule, val_loss, preds

In [16]:
def train_one_epoch_unlabelled(model, loader, val_loader,optimizer, scheduler,best_auc=0.0):
    model.train()
    total_loss = 0
    for idx,batch in tqdm(enumerate(loader)):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
        if idx%2000==0:
            val_auc,_,_=validate(model,val_loader)
            if val_auc>best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"pretrained.bin")
    return best_auc,total_loss / len(loader)

In [17]:
from transformers import get_linear_schedule_with_warmup

In [18]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    unlabelled= pd.read_csv('unlabelled_predictions.csv')
    unlabelled["text"] = unlabelled["rule"] + " [SEP] " + unlabelled["body"]
    unlabelled['label']= (unlabelled['prediction']-unlabelled['prediction'].min())/(unlabelled['prediction'].max()-unlabelled['prediction'].min()) 
    train_ds = JigsawDataset(unlabelled['text'].tolist(), unlabelled['label'].tolist(),[0]*len(unlabelled), tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_split= augmented_df.sample(frac=.2,random_state=SEED)
    val_ds = JigsawDataset(
            val_split['text'].tolist(), 
            val_split['label'].tolist(), 
            val_split['rule_id'].tolist(), 

            tokenizer, MAX_LEN
        )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = JigsawModel(MODEL_PATH).to(DEVICE)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5,eps=1e-6)
    total_steps= U_EPOCHS*len(train_loader)
    warmup_steps= 0.1*total_steps
    scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps,
            )
    best_auc=0
    for epoch in range(U_EPOCHS):
        best_auc,loss = train_one_epoch_unlabelled(model, train_loader,val_loader, optimizer, scheduler,best_auc)                

import gc

# model.cpu()
# del model, checkpoint
gc.collect()
torch.cuda.empty_cache()

In [19]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print('--------- ','FOLD: ',fold,' --------')
        all_preds = []
        val_ds = JigsawDataset(
            augmented_df.iloc[val_idx]['text'].tolist(), 
            augmented_df.iloc[val_idx]['label'].tolist(), 
            augmented_df.iloc[val_idx]['rule_id'].tolist(), 

            tokenizer, MAX_LEN
        )
    
        train_ds = JigsawDataset(
            augmented_df.iloc[tr_idx]['text'].tolist(), 
            augmented_df.iloc[tr_idx]['label'].tolist(), 
            augmented_df.iloc[tr_idx]['rule_id'].tolist(), 
            
            tokenizer, MAX_LEN,
        )
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        # Initialize classification model and load MLM pre-trained weights
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts=torch.load(f"pretrained.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
        model.load_state_dict(wts)
        
        for name, param in model.named_parameters():
            if name.startswith('base.embedding'):
                param.requires_grad = False
                # print(name)

        # model= nn.DataParallel(model)
        print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
       
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5,weight_decay=3e-2,eps=1e-6)
        total_steps= EPOCHS*len(train_loader)
        warmup_steps= 0.1*total_steps
        scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps,
            )

    
        best_auc=0
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            loss = train_one_epoch(model, train_loader, optimizer, scheduler)
            val_auc, val_loss ,val_preds = validate(model, val_loader)
            
                        
            print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_auc.bin")
    
        all_preds.append(pd.Series(val_preds))

In [20]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts=torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
            
        model.load_state_dict(wts)
        model.eval()
    
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test),[0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)

In [21]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv